# 02 — Missing Data Mechanisms

In this notebook we artificially introduce missing values into both datasets using three mechanisms:
- **MCAR** — Missing Completely At Random
- **MAR** — Missing At Random
- **MNAR** — Missing Not At Random

We test three missing rates: **10%, 30%, 50%**

Since both datasets are (mostly) complete, we control exactly which values go missing and why — giving us ground truth for evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.datasets import fetch_california_housing
import sys
sys.path.append('..')
from src.missing_generator import introduce_mcar, introduce_mar, introduce_mnar

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Datasets

In [ ]:
housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame.copy()

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
col_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]
df_adult = pd.read_csv(url, header=None, names=col_names, na_values=' ?', skipinitialspace=True)
df_adult_complete = df_adult.dropna().reset_index(drop=True)

print(f'Housing shape: {df_housing.shape}')
print(f'Adult shape (complete rows): {df_adult_complete.shape}')

## 2. California Housing — MCAR

In [ ]:
target_cols = ['MedInc', 'HouseAge', 'AveRooms']
missing_rates = [0.1, 0.3, 0.5]

fig, axes = plt.subplots(len(missing_rates), 1, figsize=(12, 10))
for ax, rate in zip(axes, missing_rates):
    df_mcar = introduce_mcar(df_housing, columns=target_cols, missing_rate=rate)
    msno.matrix(df_mcar, ax=ax, sparkline=False)
    ax.set_title(f'MCAR — missing rate {int(rate*100)}%')
plt.suptitle('California Housing — MCAR Patterns', y=1.01)
plt.tight_layout()
plt.show()

## 3. California Housing — MAR

In [ ]:
# MedInc goes missing when HouseAge is high — a plausible real-world scenario
fig, axes = plt.subplots(len(missing_rates), 1, figsize=(12, 10))
for ax, rate in zip(axes, missing_rates):
    df_mar = introduce_mar(df_housing, target_col='MedInc', condition_col='HouseAge', missing_rate=rate)
    msno.matrix(df_mar, ax=ax, sparkline=False)
    ax.set_title(f'MAR — missing rate {int(rate*100)}%')
plt.suptitle('California Housing — MAR Patterns (MedInc missing when HouseAge is high)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Verify the MAR mechanism: missing values should concentrate in high HouseAge rows
df_mar_30 = introduce_mar(df_housing, target_col='MedInc', condition_col='HouseAge', missing_rate=0.3)
missing_flag = df_mar_30['MedInc'].isna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_housing.loc[~missing_flag, 'HouseAge'], bins=30, alpha=0.6, label='Observed')
ax.hist(df_housing.loc[missing_flag, 'HouseAge'], bins=30, alpha=0.6, label='Missing')
ax.set_title('MAR verification — HouseAge distribution for observed vs missing MedInc')
ax.set_xlabel('HouseAge')
ax.legend()
plt.tight_layout()
plt.show()

## 4. California Housing — MNAR

In [ ]:
# High MedInc values are more likely to go missing
fig, axes = plt.subplots(len(missing_rates), 1, figsize=(12, 10))
for ax, rate in zip(axes, missing_rates):
    df_mnar = introduce_mnar(df_housing, target_col='MedInc', missing_rate=rate, direction='high')
    msno.matrix(df_mnar, ax=ax, sparkline=False)
    ax.set_title(f'MNAR — missing rate {int(rate*100)}%')
plt.suptitle('California Housing — MNAR Patterns (high MedInc more likely missing)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Verify MNAR: missing values should concentrate at high MedInc values
df_mnar_30 = introduce_mnar(df_housing, target_col='MedInc', missing_rate=0.3, direction='high')
missing_flag = df_mnar_30['MedInc'].isna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_housing.loc[~missing_flag, 'MedInc'], bins=30, alpha=0.6, label='Observed')
ax.hist(df_housing.loc[missing_flag, 'MedInc'], bins=30, alpha=0.6, label='Missing')
ax.set_title('MNAR verification — MedInc distribution for observed vs missing')
ax.set_xlabel('MedInc')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Adult Census — All Mechanisms

In [ ]:
# MCAR on numerical columns
num_cols_adult = ['age', 'hours_per_week', 'capital_gain']
df_adult_mcar = introduce_mcar(df_adult_complete, columns=num_cols_adult, missing_rate=0.3)

# MAR: hours_per_week missing when age is high
df_adult_mar = introduce_mar(df_adult_complete, target_col='hours_per_week', condition_col='age', missing_rate=0.3)

# MNAR: high capital_gain values more likely to go missing
df_adult_mnar = introduce_mnar(df_adult_complete, target_col='capital_gain', missing_rate=0.3, direction='high')

fig, axes = plt.subplots(3, 1, figsize=(12, 12))
for ax, (df, title) in zip(axes, [
    (df_adult_mcar, 'MCAR 30%'),
    (df_adult_mar,  'MAR 30% — hours_per_week missing when age is high'),
    (df_adult_mnar, 'MNAR 30% — high capital_gain more likely missing'),
]):
    msno.matrix(df, ax=ax, sparkline=False)
    ax.set_title(f'Adult Census — {title}')
plt.suptitle('Adult Census — Missing Patterns', y=1.01)
plt.tight_layout()
plt.show()

## 6. Save Processed Datasets

We save one version per mechanism per dataset (at 30% missing rate) for use in notebook 03.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Housing
df_housing.to_csv('../data/raw/housing.csv', index=False)
introduce_mcar(df_housing, columns=target_cols, missing_rate=0.3).to_csv('../data/processed/housing_mcar_30.csv', index=False)
introduce_mar(df_housing, target_col='MedInc', condition_col='HouseAge', missing_rate=0.3).to_csv('../data/processed/housing_mar_30.csv', index=False)
introduce_mnar(df_housing, target_col='MedInc', missing_rate=0.3, direction='high').to_csv('../data/processed/housing_mnar_30.csv', index=False)

# Adult
df_adult_complete.to_csv('../data/raw/adult.csv', index=False)
introduce_mcar(df_adult_complete, columns=num_cols_adult, missing_rate=0.3).to_csv('../data/processed/adult_mcar_30.csv', index=False)
introduce_mar(df_adult_complete, target_col='hours_per_week', condition_col='age', missing_rate=0.3).to_csv('../data/processed/adult_mar_30.csv', index=False)
introduce_mnar(df_adult_complete, target_col='capital_gain', missing_rate=0.3, direction='high').to_csv('../data/processed/adult_mnar_30.csv', index=False)

print('Datasets saved.')